# Wave 1 ASL Recognition — Colab Training

Trains the from-scratch 3D CNN on Colab's free CPU/GPU (no local PyTorch needed). Use when your local disk is too tight for the torch install.

## Workflow
1. **Local:** zip the project (minus heavy folders). Run this from the project root:
   ```powershell
   Compress-Archive -Path ml,content,scripts -DestinationPath asl_pilot.zip -Force
   ```
   Make sure your `.webm` captures sit in `ml/data/incoming/` before zipping.
2. **Colab:** open this notebook, choose `Runtime → Change runtime type → T4 GPU` (free tier), then run cells in order.
3. **Local:** drop the downloaded `model.onnx` + `labels.json` into `apps/web/public/models/`.

All cells assume Colab. None of them require resources from your local machine after upload.

## 1. Install dependencies

In [ ]:
!pip install -q torch numpy opencv-python-headless onnx onnxruntime scikit-learn tqdm pillow
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Upload the project zip
Drag-drop `asl_pilot.zip` when prompted. Expected contents: `ml/`, `content/`, and `.webm` capture files inside `ml/data/incoming/`.

In [ ]:
import os, shutil, zipfile
from google.colab import files

if os.path.exists('asl'):
    shutil.rmtree('asl')
os.makedirs('asl', exist_ok=True)
os.chdir('asl')

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
os.remove(zip_name)
print('Extracted to', os.getcwd())
print('Top-level:', sorted(os.listdir('.')))

incoming = 'ml/data/incoming'
webms = [f for f in os.listdir(incoming) if f.endswith('.webm')] if os.path.exists(incoming) else []
print(f'\nFound {len(webms)} .webm files in {incoming}')
if not webms:
    print('WARNING: No .webm files uploaded. Re-zip with your captures in ml/data/incoming/.')

## 3. Import captures → .npz

In [ ]:
!python ml/scripts/import_captures.py
import glob
npzs = glob.glob('ml/data/clips/**/*.npz', recursive=True)
print(f'\nTotal imported clips: {len(npzs)}')
from collections import Counter
per_sign = Counter(p.split('/')[-3] for p in npzs)
for s, c in sorted(per_sign.items()):
    print(f'  {s:20s}  {c}')

## 4. Build train/val/test manifest

In [ ]:
!python ml/scripts/build_manifest.py --wave1 --signer-disjoint
import json
with open('ml/data/manifest.json') as f:
    mf = json.load(f)
splits = Counter(c['split'] for c in mf['clips'])
print('Splits:', dict(splits))
print('Classes:', len(mf.get('sign_ids', [])))

## 5. Train

In [ ]:
# Use CUDA if available (T4 free tier); otherwise CPU.
# 20 epochs is a reasonable starting point — adjust based on val_acc curve.
!python ml/train.py --epochs 20 --batch-size 16 --model-version wave1-colab

## 6. Evaluate (writes docs/VALIDATION_REPORT.md)

In [ ]:
!python ml/eval.py --checkpoint ml/checkpoints/wave1-colab/best.pt --manifest ml/data/manifest.json
with open('docs/VALIDATION_REPORT.md') as f:
    print(f.read())

## 7. Export ONNX (single self-contained file)

In [ ]:
!python ml/export_onnx.py --checkpoint ml/checkpoints/wave1-colab/best.pt
import os
for f in ['model.onnx', 'labels.json', 'model_meta.json']:
    p = f'ml/exports/{f}'
    if os.path.exists(p):
        print(f'  {f}: {os.path.getsize(p)/1e6:.2f} MB')

## 8. Download artifacts back to your machine
These three files go into `apps/web/public/models/` locally, and `VALIDATION_REPORT.md` goes into `docs/`.

In [ ]:
from google.colab import files
for f in ['ml/exports/model.onnx', 'ml/exports/labels.json', 'ml/exports/model_meta.json', 'docs/VALIDATION_REPORT.md']:
    if os.path.exists(f):
        files.download(f)
    else:
        print('missing:', f)